## 1. Load Environment Configuration

Loads values from the project `.env` file into this notebook's Python process. It also sets a descriptive HTTP `User-Agent` for job-page requests.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ.setdefault("USER_AGENT", "cold-email-generator/0.1")

## 2. List Available Groq Models

Calls Groq's models endpoint and prints the model IDs available to this API account. Use an active chat model ID in the following configuration cell.

In [ ]:
from groq import Groq

client = Groq()
for model in client.models.list().data:
    print(model.id)

## 3. Configure the Chat Model

Creates the LangChain Groq chat model using the selected active model ID and deterministic temperature. Later chains reuse this `llm` object.

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
)

## 4. Verify the Notebook Interpreter

Prints the Python executable used by the selected notebook kernel. Confirm it points to this project's `.venv` so the notebook can access the installed dependencies.

In [ ]:
import sys
print(sys.executable)

## 5. Fetch the Job Posting

Uses `WebBaseLoader` to request the configured careers-page URL, converts the page into text, and stores it in `p_data` for extraction.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://careers.nike.com/retail-associate-pt-nike-hershey/job/R-93670")
p_data=loader.load().pop().page_content
print(p_data)

## 6. Define the Job-Details Prompt

Defines the extraction instructions and creates a `PromptTemplate` that combines those instructions with the scraped page text.

In [ ]:
from langchain_core.prompts import PromptTemplate

# Define a template to extract job posting details from scraped content
template = """
### SCRAPPED TEXT FROM WEBSITE:
{p_data}

### INSTRUCTION:
The scrapped text is from the career's page of a website. 
Your job is to extract the job posting details and return them in JSON format containing the following keys: 
- 'role': Job title/position
- 'experience': Experience required for the role
- 'skills': A list of the skills required
- 'description': A brief description of the job role

Only return the valid JSON.

### VALID JSON:
"""

# Create the prompt template using the defined string
prompt_template = PromptTemplate.from_template(template)

# Print the created prompt template
#print(prompt_template)
filled_prompt = prompt_template.format(p_data=p_data)
#print(filled_prompt)

## 7. Extract Job Details with the LLM

Combines the extraction prompt and chat model into a chain, sends the scraped job text, and prints the model's structured response.

In [ ]:
chain_extract=prompt_template | llm
res=chain_extract.invoke({'p_data':p_data})
print(res.content)

## 8. Check the Model Response Type

Displays the Python type of the raw response content. This is a quick diagnostic; it does not validate or transform the response.

In [ ]:
type(res.content)

## 9. Parse the Response as JSON

Uses `JsonOutputParser` to convert the model's JSON text into a Python object named `json_res` for later portfolio matching.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
jp=JsonOutputParser()
json_res=jp.parse(res.content)
type(json_res)

## 10. Draft Prompt Notes (Markdown Only)

The following cell contains a draft prompt written as Markdown, so it is documentation and will not execute. The active extraction prompt is defined in the earlier Python cell.

from langchain_core.prompts import PromptTemplate as pt
prompt_extract=pt.from_template(
    """
    ### SCRAPPED TEXT FROM WEBSITE:
    {p_data}
    ### INSTRUCION:
    The scrapped text is from the career's page of a website.
    Your job is to extract the job posting and return them in JSON format containing following keys: 'role','experience','skills',and 'description'.
    Only return the valid JSON.
    ### VALID JSON (NO PREAMBLE):
    """
)

## 11. Load the Portfolio CSV

Reads the portfolio records from `data/portfolios.csv` into a pandas DataFrame. These records are the source data for semantic portfolio matching.

In [ ]:
import pandas as pd
df=pd.read_csv(r'data\portfolios.csv')
df

## 12. Create or Populate the Chroma Collection

Opens a persistent Chroma collection and, when it is empty, adds each portfolio tech stack with its link and a unique ID.

In [ ]:
import uuid
import chromadb as cdb
client=cdb.PersistentClient('vectorstore')#on disk
collection=client.get_or_create_collection(name='portfolio')
if not collection.count():
    for _,row in df.iterrows():
        collection.add(documents=row["Techstack"],metadatas={"links":row["Links"]},ids=[str(uuid.uuid4())])
                                    

## 13. Example Portfolio Query (Markdown Only)

The following cell shows a sample Chroma query, but it is Markdown rather than Python, so it will not run. The active query uses the extracted job skills in a later code cell.

links=collection.query(query_texts=["Experience in Python","Expertise in React"],n_results=2).get('metadatas',[])
links

## 14. Select the Parsed Job Data

Assigns the parsed job dictionary to `job` and displays its required skills for use in the portfolio search.

In [ ]:
job=json_res
job['skills']

## 15. Retrieve Relevant Portfolio Links

Queries Chroma with the job's required skills and stores the matching portfolio metadata in `links` for the email prompt.

In [ ]:
links=collection.query(query_texts=job['skills'],n_results=2).get('metadatas',[])
links

## 16. Generate the Cold Email

Builds an email prompt from the job details and matched portfolio links, sends it to the chat model, and prints the generated draft.

In [ ]:
email_prompt=PromptTemplate.from_template(
    """
    ### JOB DESCRIPTION:
        {job_description}

    ### INSTRUCTION:
    Your job is to write a cold email to the client regarding the job mentioned above describing the capabilities in fulfilling their needs.
    Also add the most relevant ones from the following links to showcase ABC's portfolio:{link_list}
    Remember you are Bhupendra Kumar Swar,general manager of ABC
    Do not provide the preamble.
    ### EMAIL (NO PREAMBLE):
    """
)
chain_email=email_prompt | llm
e_res=chain_email.invoke({'job_description':str(job),'link_list':links})
print(e_res.content)

## 17. Empty Cell

This code cell is currently blank and has no effect. It is available for future experiments.